In [ ]:
from dotenv import load_dotenv
load_dotenv()




: 

In [2]:
from langchain_core.tools import tool


@tool("add_numbers", return_direct=True, description="Add two numbers together.")
def ad_nb(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b

@tool()
def get_current_weather(location: str) -> str:
    """Get the current weather in a given location."""
    return f"The current weather in {location} is sunny with a high of 25°C."

@tool()
def get_whr(location: str,units:str = "credits", include_forecast: bool = False) -> str:
    """Get the weather in a given location."""
    forecast = " The forecast for tomorrow is cloudy with a chance of rain." if include_forecast else ""
    temp = 25 if units == "credits" else 77
    result = f"The current weather in {location} is sunny with a high of {temp}°{units}.{forecast}"
    if include_forecast:
        result += " The forecast for tomorrow is cloudy with a chance of rain."
    return result

In [3]:
from pydantic import BaseModel, Field
from typing import Literal

class WeatherToolInput(BaseModel):
    location: str = Field(..., 
                          description="The location to get the weather for.")
    units: Literal["celsius", "imperial"] = Field("celsius", 
                                                 description="The units to use for the weather data.")
    include_forecast: bool = Field(False, 
                                   description="Whether to include the forecast for tomorrow.")

@tool(args_schema=WeatherToolInput)
def get_weather(location: str,units:str = "celsius", include_forecast: bool = False) -> str:
    """Get the weather in a given location."""
    temp = 25 if units == "celsius" else 77
    result = f"The current weather in {location}:{temp} degrees {units[0].upper()}."
    if include_forecast:
        result += " The forecast for tomorrow is cloudy with a chance of rain."
    return result

In [4]:
get_weather.invoke({"location": "New York", "include_forecast":True})

'The current weather in New York:25 degrees C. The forecast for tomorrow is cloudy with a chance of rain.'

In [6]:
from langchain.agents import create_agent
from langchain.messages import SystemMessage, HumanMessage

agent = create_agent(model='deepseek-chat',
                     tools=[get_weather])
for token, metadata in agent.stream(
    {'messages':[HumanMessage(content="接下来深圳几天天气如何")]},
    stream_mode="messages"
):
    print(token.content,end='',flush=True)


好的，让我查一下深圳的天气情况，包括今明几天的预报。The current weather in 深圳:25 degrees C. The forecast for tomorrow is cloudy with a chance of rain.以下是深圳这几天的天气情况：

**今天：**
- 🌡️ 当前温度：**25°C**
- 天气状况：晴天（数据中未特别说明，但当前温度适宜）

**明天（预报）：**
- ☁️ **多云，有降雨可能**
- 建议出门带伞，以防阵雨

---

**温馨提示：**
- 深圳目前气温舒适，约 25°C，适合外出活动。
- 明天可能有雨，建议关注实时天气变化。

如果你需要更详细的逐日预报或未来几天的天气趋势，也可以告诉我，我可以用其他方式帮你查询哦！😊

In [9]:
from langchain_tavily.tavily_search import TavilySearch
from langgraph.channels import Topic


search_tool = TavilySearch(
    max_result = 5,
    Topic = 'general',
)

search_tool.invoke('蒸蚌是什么')

{'error': ValueError('Error 401: Unauthorized: missing or invalid API key.')}